In [3]:
#Q1. Bayes’ Theorem Basics (Math + Code)
from collections import Counter
emails = [
    ("Win money now", "Spam"),
    ("Lowest price guaranteed", "Spam"),
    ("Cheap meds available", "Spam"),
    ("Hello friend how are you", "Ham"),
    ("Let’s have lunch tomorrow", "Ham"),
    ("Meeting schedule attached", "Ham"),
    ("Win a free lottery ticket", "Spam"),
    ("See you at the conference", "Ham"),
    ("Project deadline reminder", "Ham"),
    ("Cheap loans available", "Spam")
]
def tokenize(text):
    return text.lower().split()
spam_words, ham_words = [], []
for text, label in emails:
    if label == "Spam":
        spam_words.extend(tokenize(text))
    else:
        ham_words.extend(tokenize(text))
spam_count = Counter(spam_words)
ham_count = Counter(ham_words)
V = len(set(spam_words + ham_words))
spam_total, ham_total = len(spam_words), len(ham_words)
def word_prob(word, label):
    if label == "Spam":
        return (spam_count[word] + 1) / (spam_total + V)
    else:
        return (ham_count[word] + 1) / (ham_total + V)
def classify(text):
    words = tokenize(text)
    p_spam = 0.5
    p_ham = 0.5

    for w in words:
        p_spam *= word_prob(w, "Spam")
        p_ham *= word_prob(w, "Ham")

    norm = p_spam + p_ham
    return {"Spam": p_spam / norm, "Ham": p_ham / norm}
print(classify("cheap price now"))

{'Spam': 0.9346071511526928, 'Ham': 0.06539284884730717}


In [5]:
#Q2. Spam Classi=cation (Text → Features → Naïve Bayes)

import re
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
def preprocess(text):
    text = text.lower()
    words = re.findall(r'\b[a-z]+\b', text)
    return [w for w in words if w not in stop_words]
print(preprocess("Hello friend, how are you?"))
from collections import Counter
all_words = []
for text, label in emails:
    all_words.extend(preprocess(text))
vocab = set(all_words)
print("Vocabulary:", vocab)
def extract_features(text):
    words = preprocess(text)
    return Counter(words)
print(extract_features("Win money now"))

spam_texts = [text for text, label in emails if label == "Spam"]

ham_texts  = [text for text, label in emails if label == "Ham"]
spam_words = []
for text in spam_texts:
    spam_words.extend(preprocess(text))
ham_words = []
for text in ham_texts:
    ham_words.extend(preprocess(text))
spam_count = Counter(spam_words)
ham_count = Counter(ham_words)
V = len(vocab)
spam_total = len(spam_words)
ham_total = len(ham_words)
def word_prob(word, label):
    if label == "Spam":
        return (spam_count[word] + 1) / (spam_total + V)
    else:
        return (ham_count[word] + 1) / (ham_total + V)
def classify(text):
    words = preprocess(text)
    p_spam = len(spam_texts) / len(emails)  # prior
    p_ham  = len(ham_texts) / len(emails)   # prior

    for w in words:
        p_spam *= word_prob(w, "Spam")
        p_ham  *= word_prob(w, "Ham")

    norm = p_spam + p_ham
    return {"Spam": p_spam / norm, "Ham": p_ham /norm}

correct = 0
for text, label in emails:
    result = classify(text)
    predicted = max(result, key=result.get)
    if predicted == label:
        correct += 1
    print(f"Email: {text:30} | Predicted: {predicted} | Actual: {label}")
accuracy = correct / len(emails)
print("Accuracy:", accuracy)

['hello', 'friend']
Vocabulary: {'attached', 'friend', 'ticket', 'lottery', 'price', 'conference', 'available', 'meds', 'lowest', 'schedule', 'loans', 'let', 'lunch', 'hello', 'project', 'guaranteed', 'free', 'win', 'cheap', 'deadline', 'meeting', 'reminder', 'see', 'money', 'tomorrow'}
Counter({'win': 1, 'money': 1})
Email: Win money now                  | Predicted: Spam | Actual: Spam
Email: Lowest price guaranteed        | Predicted: Spam | Actual: Spam
Email: Cheap meds available           | Predicted: Spam | Actual: Spam
Email: Hello friend how are you       | Predicted: Ham | Actual: Ham
Email: Let’s have lunch tomorrow      | Predicted: Ham | Actual: Ham
Email: Meeting schedule attached      | Predicted: Ham | Actual: Ham
Email: Win a free lottery ticket      | Predicted: Spam | Actual: Spam
Email: See you at the conference      | Predicted: Ham | Actual: Ham
Email: Project deadline reminder      | Predicted: Ham | Actual: Ham
Email: Cheap loans available          | Predicted: 

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
#Q3. Now use sklearn.naive_bayes.MultinomialNB and compare performance.

from sklearn.feature_extraction.text import CountVectorizer
texts, labels = zip(*emails)  # separate data and labels
vectorizer = CountVectorizer(stop_words="english")
X = vectorizer.fit_transform(texts)  # features
y = labels
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
clf = MultinomialNB()
clf.fit(X, y)
# Predictions
y_pred = clf.predict(X)
print("Accuracy:", accuracy_score(y, y_pred))
print(classification_report(y, y_pred))
test_email = ["cheap price now"]
X_test = vectorizer.transform(test_email)
print("Prediction:", clf.predict(X_test)[0])
print("Probabilities:", clf.predict_proba(X_test))

Accuracy: 1.0
              precision    recall  f1-score   support

         Ham       1.00      1.00      1.00         5
        Spam       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

Prediction: Spam
Probabilities: [[0.16360116 0.83639884]]
